# 🔍 News Bias Detector — Colab Launcher

Run all cells in order. The last cell opens Marimo via a public URL using `localtunnel` or `ngrok`.

**Pipeline:**
```
RSS feeds → Sentence Embeddings → PCA → EVōC (2 pre-built layers)
                                                    ↓
Query → encode → cosine routing (per layer top-k) → deep cluster on-the-fly → NLI bias labels
```

In [ ]:
# ── Cell 1: Install packages ──────────────────────────────────────────────────
!pip install -q marimo evoc sentence-transformers scikit-learn plotly \
    transformers torch keybert feedparser requests

In [ ]:
# ── Cell 2: Download the Marimo app from this notebook's files ────────────────
# If you uploaded bias_detector.py to Colab, skip this cell.
# Otherwise paste the full content of bias_detector.py here or upload via the
# Colab file browser (left sidebar → Files → Upload).
import os
print('bias_detector.py present:', os.path.exists('bias_detector.py'))

In [ ]:
# ── Cell 3: Install tunnel (choose ONE of the options below) ──────────────────

# Option A — localtunnel (no account needed)
!npm install -g localtunnel 2>/dev/null | tail -1

# Option B — ngrok (needs free account + authtoken)
# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_TOKEN_HERE')

In [ ]:
# -- Cell 4: Launch Marimo editor in the background --
import subprocess, threading, time, requests as req

PORT = 2718

# Start Marimo editor
proc = subprocess.Popen(
    ['python', '-m', 'marimo', 'edit', 'bias_detector.py',
     '--host', '0.0.0.0', '--port', str(PORT), '--no-token'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

# Wait for server to start
for _ in range(20):
    try:
        if req.get(f'http://localhost:{PORT}', timeout=2).status_code == 200:
            print('✅ Marimo editor started')
            break
    except Exception:
        pass
    time.sleep(2)

print(f'Marimo editor running on port {PORT}')

In [ ]:
# ── Cell 5A: Expose via localtunnel ───────────────────────────────────────────
import subprocess, re

tunnel_proc = subprocess.Popen(
    ['lt', '--port', str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

for line in tunnel_proc.stdout:
    match = re.search(r'https://[\w-]+\.loca\.lt', line)
    if match:
        url = match.group(0)
        print(f'🌐 Open your Bias Detector: {url}')
        print('   (If prompted for a password, visit https://loca.lt/mytunnelpassword)')
        break

In [ ]:
# ── Cell 5B (alternative): Expose via ngrok ───────────────────────────────────
# Uncomment if you prefer ngrok

# from pyngrok import ngrok
# public_url = ngrok.connect(PORT)
# print(f'🌐 Open your Bias Detector: {public_url}')

## Usage walkthrough

1. **Select RSS sources** and click **Fetch Articles**
2. Click **Build Cluster Index** — EVōC builds 2 hierarchical layers from the embeddings
3. Explore the t-SNE scatter to see how articles cluster
4. Enter a **query** (e.g. `"climate change"`) and click **Analyse Bias**
5. The system:
   - Embeds your query
   - Routes through Layer 0 (fine) and Layer 1 (coarse) by cosine similarity, keeping top-k clusters per layer
   - Re-ranks all candidate articles by direct cosine similarity
   - Sub-clusters the final candidates on-the-fly with a fresh EVōC pass
   - Runs **NLI zero-shot classification** to detect bias labels per article
6. View charts, source summaries, and download the CSV

### Architecture diagram
```
Titles
  │
  ▼
SentenceTransformer  →  384-dim embeddings (L2-normed)
  │
  ▼
PCA (→ 64-dim)        reduces noise, speeds EVōC
  │
  ▼
EVōC (max_layers=2)   pre-built at index time
  ├── Layer 0  (fine clusters, ~N/30 articles each)
  └── Layer 1  (coarse clusters)
         │
  ── QUERY ──────────────────────────────────────────────
         │
  encode query  →  cosine similarity
         ├── Layer 0 centroids: top-k clusters → candidate articles
         ├── Layer 1 centroids: top-k clusters → more candidates
         ├── Noise articles: direct cosine top-k
         ▼
  rank all candidates by cosine(query, article_embedding)
         │
  EVōC on-the-fly (1 layer) on top-N candidates
         │
  NLI zero-shot classification per article title
         ▼
  Results: ranked table + bias charts + source breakdown
```
